# SahiTol e-waste classifier v2 — isolated Colab experiment

This notebook is deliberately isolated from the Android app and its bundled model. It does not grant permission to promote a model. Run it only after copying `experiments/e_waste_model_v2/` into Google Drive or cloning the reviewed repository.

**Safety rule:** sources marked `HOLD_*` or `PENDING_*` in `source_registry.json` must not enter the approved model. A separately named, owner-authorized quarantined benchmark may inspect Roboflow/Kaggle archives, but it must never change the APK, approved float32 model, or release evidence.

In [ ]:
# Mount a Drive folder that contains the reviewed experiment directory.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
EXPERIMENT = Path('/content/drive/MyDrive/SahiTol/experiments/e_waste_model_v2')
assert (EXPERIMENT / 'source_registry.json').exists(), 'Copy the reviewed experiment folder to Drive first.'
print(EXPERIMENT)

In [ ]:
# Validate the six-source registry before downloading or processing anything.
!python {EXPERIMENT / 'scripts' / 'validate_registry.py'}

## Quarantined Roboflow/Kaggle inventory — first gate only\n\nUse this only for the owner-authorized, non-product comparison. Download each provider archive through its own site/API without placing any token in this notebook or repository. Put the ZIP at the declared path, run one source at a time, and paste the JSON report here. Do **not** train yet: the observed provider labels determine whether an automatic direct mapping exists.

In [ ]:
# Set exactly one source and point ARCHIVE to its downloaded ZIP.\n# Roboflow exports may require an owner-side Roboflow login/API key; never paste that key here.\n# Kaggle downloads may require an owner-side Kaggle login/token; never paste that token here.\nSOURCE_ID = 'ROBOFLOW_EWASTE_2025'  # then repeat separately with KAGGLE_AKSHAT_EWASTE\nARCHIVE = EXPERIMENT / 'data' / 'raw' / SOURCE_ID / 'provider_download.zip'\nOUTPUT = EXPERIMENT / 'data' / 'prepared' / 'quarantine' / SOURCE_ID\nassert ARCHIVE.is_file(), f'Download the provider ZIP first: {ARCHIVE}'\nassert not OUTPUT.exists(), f'Remove nothing automatically. Choose a new output folder if a prior inventory exists: {OUTPUT}'\n!python {EXPERIMENT / 'scripts' / 'inventory_quarantined_archive.py'} --source-id {SOURCE_ID} --archive {ARCHIVE} --output-dir {OUTPUT}

## Download and asset-ledger rule

Use provider APIs/downloads only for sources currently eligible in the registry. Store archives under `data/raw/<source-id>/` and create one ledger row per accepted image. The ledger must retain: source ID, original URL, exact licence, attribution, SHA-256, provider label, approved SahiTol mapping, object/sequence group, and split.

Do not use Kaggle or Zenodo MMEWaste merely because their download links work. They are held until their rights/scope blockers have written resolution. No manually drawn boxes or invented labels are required or permitted.

In [ ]:
# Install only the experiment dependencies after raw, individually auditable assets exist.
!pip -q install tensorflow==2.16.1 scikit-learn==1.5.1 pandas==2.2.2 pillow==10.4.0 imagehash==4.3.1

# The following assertion intentionally blocks a premature training run.
ledger = EXPERIMENT / 'data' / 'prepared' / 'asset_ledger.csv'
assert ledger.exists(), 'No asset ledger yet. Complete automated source ingestion and audit before training.'

## Required evaluation before any promotion

Use grouped 70/15/15 splits; use validation only for checkpoint and threshold selection; do not open the test set during tuning. The final report must include exact class/source counts, accuracy, macro-F1, per-class precision/recall/F1, confusion matrix, threshold coverage/abstention, unsafe confusion review, model size, and LiteRT parity.

A new `.tflite` file is an experiment output only. It must not replace `data/curated/model/classifier.tflite` or Android assets until the owner explicitly approves promotion and device verification succeeds.